# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)


Setting up connection to the FlyRank warehouse on Hugging Face using DuckDB.



In [39]:
%pip install -q duckdb huggingface_hub

import duckdb
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")
con.execute(f"CREATE SECRET hf_token (TYPE HUGGINGFACE, TOKEN '{hf_token}');")

What one row means for my lane: One row = one content page's performance on one specific day (from fact_content_daily_performance, grain = report_date + client_hash_id + content_hash_id).

Which table(s): dim_content joined to fact_content_daily_performance on content_hash_id.

Time window: Mid-panel month, month=2026-03, for development — not the final month, which is a sealed test period.

What I'd predict/rank: A refresh-priority score per page — ranking by likelihood of being worth a content review (declining visibility + real traffic).

What I deliberately exclude: Product decision flags (health_score, priority_score, action_type) — these are the product's own outputs, not raw signals, and would leak the answer.

##Query 1: checking the grain — confirming one row really is one page-day.



In [40]:
q1 = con.execute("""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) as row_count
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    GROUP BY 1,2,3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print(q1)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Empty DataFrame
Columns: [report_date, client_hash_id, content_hash_id, row_count]
Index: []


##Result: Empty DataFrame — zero rows had a duplicate grain. This confirms the grain claim: one row genuinely equals one page-day (report_date + client_hash_id + content_hash_id), with no unexpected duplicates.



##Query 2: row count and date span for this month's slice.



In [41]:
  q2 = con.execute("""
    SELECT COUNT(*) as total_rows, MIN(report_date) as min_date, MAX(report_date) as max_date
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
""").df()
print(q2)

   total_rows   min_date   max_date
0     9841378 2026-03-01 2026-03-31


##Result: 9,841,378 rows spanning March 1, 2026 to March 31, 2026 — the full month, as expected for month=2026-03.



##Query 3: availability check using IS TRUE.



In [42]:
q3 = con.execute("""
    SELECT COUNT(*) as total,
           SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) as ga4_available_rows
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
""").df()
print(q3)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

     total  ga4_available_rows
0  9841378            413966.0


##Result: Of 9,841,378 total rows, only 413,966 have GA4 data available (about 4.2%). This confirms GA4 (Analytics) coverage is sparse compared to the full dataset — most rows likely have GSC (Search Console) data only. This is an important availability constraint: any feature relying on GA4 fields (like engagement/sessions) will only be usable for a small fraction of pages.


# Features (knowable at decision moment):

avg_impressions — past-window observed signal (GSC data), not a future outcome.
avg_clicks — past-window observed signal, same reasoning.
avg_ctr — derived purely from past clicks/impressions within the window, no future data used.
avg_position — past-window observed ranking signal.
days_with_data — simply counts how many days had any data in the window; no future leakage.

Label (what I'd predict/rank): A refresh-priority score — not present as a column yet; this will be derived in modeling weeks from a future outcome window, kept separate from these features.
Context (not used as a feature, but useful background): content_hash_id — identifier only, used for grouping/joining, not as a predictive signal itself.
Excluded: Any FlyRank product decision flags (health_score, priority_score, action_type) — these are the product's own outputs, not raw observable signals, and using them would leak the answer into my features.

In [43]:
features_q = con.execute("""
    SELECT
        content_hash_id,
        AVG(gsc_impressions) as avg_impressions,
        AVG(gsc_clicks) as avg_clicks,
        AVG(CASE WHEN gsc_impressions > 0 THEN gsc_clicks*1.0/gsc_impressions ELSE 0 END) as avg_ctr,
        AVG(gsc_avg_position) as avg_position,
        COUNT(*) as days_with_data
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    GROUP BY content_hash_id
""").df()
print(features_q.head(10))
print(f"\nTotal unique pages: {len(features_q)}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

            content_hash_id  avg_impressions  avg_clicks   avg_ctr  \
0  content_d0dff76c889de68f         5.838710    0.000000  0.000000   
1  content_67741cce996cfafa         1.483871    0.032258  0.008065   
2  content_2e6360ad20fd7107        29.000000    0.032258  0.001613   
3  content_ac8663da7484669a         1.096774    0.000000  0.000000   
4  content_65c50dfe9d87a585       100.258065    0.000000  0.000000   
5  content_d49a012dcb924e31        10.612903    0.000000  0.000000   
6  content_614baf2af4330bd7        24.903226    0.032258  0.000922   
7  content_4dc944b7d0b65ecc         4.322581    0.000000  0.000000   
8  content_4a1ca0fa5c177e0c         0.451613    0.000000  0.000000   
9  content_225dc9235023be5f        15.741935    0.032258  0.001613   

   avg_position  days_with_data  
0      5.147402              31  
1      4.828125              31  
2      5.145765              31  
3      4.909314              31  
4      6.969536              31  
5      5.177774          

Result: Built a 5-feature frame across 331,437 unique pages for March 2026 — avg_impressions, avg_clicks, avg_ctr, avg_position, and days_with_data, all derived purely from within-window observed GSC signals. Each page has up to 31 days_with_data (matching the full month), confirming good coverage for this feature set.

##Verification query:
Confirming the feature frame has no broken identifiers (null content_hash_id) and that days_with_data falls within a valid range (1 to 31, since this is a 31-day month).



In [44]:
# Verify: no missing content_hash_id, and days_with_data is within valid range (1-31)
check = con.execute("""
    SELECT
        MIN(days_with_data) as min_days,
        MAX(days_with_data) as max_days,
        SUM(CASE WHEN content_hash_id IS NULL THEN 1 ELSE 0 END) as null_ids
    FROM (
        SELECT content_hash_id, COUNT(*) as days_with_data
        FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
        GROUP BY content_hash_id
    )
""").df()
print(check)

   min_days  max_days  null_ids
0         1        31       0.0


Result: min_days = 1, max_days = 31, null_ids = 0. This confirms every page has between 1 and 31 valid days of data within the month (no page exceeds the month's actual day count), and there are zero broken/missing content identifiers. The feature frame is structurally sound.



##Data limits:
 This data can't tell me why a page is declining or growing — only that it is (correlational, not causal). GA4 (Analytics) data is sparse: only about 4.2% of rows had ga4_data_available IS TRUE in this month, so any feature relying on GA4 fields (engagement, sessions) would only be usable for a small fraction of pages — most rows are GSC-only. I'm also using a single mid-panel month (March 2026), which may not capture seasonal patterns present in other months. Finally, since I'm developing on a fixed monthly window, features near the start/end of the month have fewer days of history than pages observed for the full month, which could slightly bias volume-based features for partial-month pages

In [45]:
ga4_check = con.execute("""
    SELECT COUNT(*) as total,
           SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) as ga4_available_rows,
           ROUND(100.0 * SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) / COUNT(*), 2) as pct_ga4_available
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
""").df()
print(ga4_check)

     total  ga4_available_rows  pct_ga4_available
0  9841378            413966.0               4.21
